In [2]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [3]:
#Data cleaning

movies_metadata = pd.read_csv("dataset/movies/movies_metadata.csv")
movies_metadata = movies_metadata[['title', 'belongs_to_collection', 'id','genres']]

#rename column 'id' to 'movieId' 
movies_metadata = movies_metadata.rename(columns={"id": "movieId", 'belongs_to_collection': 'collection'})

#Convert data type of 'movieId' from string int64 and making any 'movieId' thats not a number to NaN then dropping them
movies_metadata['movieId'] = pd.to_numeric(movies_metadata['movieId'], errors='coerce')
movies_metadata = movies_metadata.dropna(subset=['movieId'])
movies_metadata['movieId'] = movies_metadata['movieId'].astype('int64')

#Extract the different genres and put them into an array without their corresponding ids
movies_metadata['genres'] = movies_metadata['genres'].apply(lambda x: [item['name'] for item in ast.literal_eval(x)])

#Extract the collection the movie belongs to if any
movies_metadata['collection'] = movies_metadata['collection'].apply(lambda x: ast.literal_eval(x)['name'] if pd.notna(x) and x != '' else '')

/var/folders/7k/s2jvnzk97s987g3tmxp90kdw0000gn/T/ipykernel_62353/1847997518.py:3: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_metadata = pd.read_csv("dataset/movies/movies_metadata.csv")


In [4]:
#Create the movie matrix

movie_matrix = movies_metadata
movie_matrix['title'] = movie_matrix['title'].str.lower()

movie_matrix['genres'] = movie_matrix['genres'].apply(lambda genres: [genre.lower() for genre in genres])
movie_matrix['genres'] = movie_matrix['genres'].apply(lambda x: ', '.join(x))

movie_matrix['collection'] = movie_matrix['collection'].apply(lambda x: x.lower())

movie_matrix2 = movie_matrix
movie_matrix2 = movie_matrix2.drop('movieId', axis=1)

movie_matrix2['data'] = movie_matrix2[movie_matrix2.columns[1:]].apply(lambda x: ' '.join(x.dropna().astype(str).str.replace(',', '')), axis=1)

In [5]:
vectorizer = CountVectorizer()
vectorized = vectorizer.fit_transform(movie_matrix2['data'])
similarities = cosine_similarity(vectorized)
movie_matrix = pd.DataFrame(similarities, columns=movie_matrix['title'], index=movie_matrix['title'])

In [10]:
movie_matrix.head()

title,toy story,jumanji,grumpier old men,waiting to exhale,father of the bride part ii,heat,sabrina,tom and huck,sudden death,goldeneye,...,house of horrors,shadow of the blair witch,the burkittsville 7,caged heat 3000,robin hood,subdue,century of birthing,betrayal,satan triumphant,queerama
title,,,,,,,,,,,,,,,,,,,,,
toy story,1.000000,0.235702,0.333333,0.235702,0.333333,0.000000,0.288675,0.204124,0.000000,0.166667,...,0.0,0.0,0.0,0.0,0.000000,0.288675,0.00000,0.000000,0.0,0.0
jumanji,0.235702,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.577350,0.333333,0.235702,...,0.0,0.0,0.0,0.0,0.000000,0.408248,0.00000,0.000000,0.0,0.0
grumpier old men,0.333333,0.000000,1.000000,0.471405,0.333333,0.000000,0.577350,0.000000,0.000000,0.166667,...,0.0,0.0,0.0,0.0,0.235702,0.000000,0.00000,0.000000,0.0,0.0
waiting to exhale,0.235702,0.000000,0.471405,1.000000,0.235702,0.288675,0.816497,0.288675,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.666667,0.408248,0.57735,0.333333,0.0,0.0
father of the bride part ii,0.333333,0.000000,0.333333,0.235702,1.000000,0.000000,0.288675,0.000000,0.000000,0.166667,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.0


In [ ]:
import pickle

#saves the movie_matrix to a file
movie_matrix.to_pickle('movie_matrix.pkl')


In [ ]:
#Movie Recommendation
input_movie = 'jumanji'
recommendations = movie_matrix[input_movie.lower()].nlargest(11)
recommendations = recommendations[recommendations.index != input_movie.lower()]
print(recommendations)

title
the indian in the cupboard                  1.0
the wizard of oz                            1.0
labyrinth                                   1.0
return to oz                                1.0
clash of the titans                         1.0
journey to the center of the earth          1.0
peter pan                                   1.0
jason and the argonauts                     1.0
the new adventures of pippi longstocking    1.0
the brothers lionheart                      1.0
Name: jumanji, dtype: float64
